# Otzi Reanalysis Notebook (Template)\n\nUse this notebook as an execution ledger from this point onward.\nHeavy compute stays in shell/Docker; this notebook records commands, outputs, and quick summaries.

In [ ]:
from pathlib import Path
import subprocess, shlex, datetime, os

# Resolve repository root from current location.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

WD = ROOT / 'mapping' / 'tst'
RAW_DATA_DIR = Path(os.environ.get('RAW_DATA_DIR', '/path/to/raw_data'))
ANALYSIS_DIR = Path(os.environ.get('ANALYSIS_DIR', '/path/to/analysis'))
OUTPUT_DIR = Path(os.environ.get('OUTPUT_DIR', str(Path.home())))

RUN_TS = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
print('ROOT:', ROOT)
print('RUN_TS:', RUN_TS)


In [ ]:
def run(cmd, cwd=WD, tee=None):\n    print('$', cmd)\n    p = subprocess.run(cmd, shell=True, cwd=str(cwd), text=True, capture_output=True)\n    out = p.stdout + p.stderr\n    if tee:\n        Path(tee).write_text(out)\n    print(out[-4000:] if len(out) > 4000 else out)\n    if p.returncode != 0:\n        raise RuntimeError(f'command failed ({p.returncode})')\n    return out

## 1) Record Inputs

In [ ]:
BAM = str(RAW_DATA_DIR / 'iceman.oetzi.UDG_D2049_combined.mapped_rmdup.bam')
REF = str(ROOT / 'mapping' / 'index' / 'hg38p14DH3630O.fa')
print(BAM)
print(REF)


## 2) Prefix Composition

In [ ]:
cmd = r"samtools view {bam} | awk '{{p=substr($1,1,1); c[p]++}} END{{print \"M\",c[\"M\"]+0; print \"F\",c[\"F\"]+0; print \"R\",c[\"R\"]+0; print \"other\", NR-(c[\"M\"]+c[\"F\"]+c[\"R\"])}}'".format(bam=BAM)\nrun(cmd)

## 3) DeDup Pre/Post Comparison (Example)

In [ ]:
PRE = 'iceman.oetzi.UDG_D2049_combined.mapped_rmdup.pair.prim.Nsort.bam'\nPOST = '/mnt/mirrored/iceman-reanalysis/dedup_out50/iceman.oetzi.UDG_D2049_combined.mapped_rmdup.pair.prim_rmdup.Nsort.bam'\ncmd = f"join -t$'\\t' -v1 <(samtools view {PRE}) <(samtools view {POST}) | cut -c1 | uniq -c"\nrun('bash -lc ' + shlex.quote(cmd))

## 4) Removed-read BAM Construction (Example)

In [ ]:
REM_SAM = '/mnt/mirrored/iceman-reanalysis/dedup_out50/iceman.oetzi.UDG_D2049_combined.mapped_rmdup.pair.prim_rmdup.Nsort.removed.sam'\nREM_BAM = '/mnt/mirrored/iceman-reanalysis/dedup_out50/iceman.oetzi.UDG_D2049_combined.mapped_rmdup.pair.prim_rmdup.Nsort.removed.bam'\nrun(f'samtools view -H {PRE} > {REM_SAM}')\ncmd = f"join -t$'\\t' -v1 <(samtools view {PRE}) <(samtools view {POST}) >> {REM_SAM}"\nrun('bash -lc ' + shlex.quote(cmd))\nrun(f'samtools view -b -o {REM_BAM} {REM_SAM}')\nrun(f'samtools flagstat {REM_BAM}')

## 5) DeepVariant Run Ledger

In [ ]:
DV_CMD = f"""
docker run --rm --user "$(id -u):$(id -g)" \
  -v {ANALYSIS_DIR}:/input \
  -v {ROOT}:/repo \
  -v {OUTPUT_DIR}:/output \
  google/deepvariant:1.10.0 \
  /opt/deepvariant/bin/run_deepvariant \
  --model_type=WGS \
  --ref=/repo/mapping/index/hg38p14DH3630O.fa \
  --reads=/input/iceman.oetzi.UDG_merge_combined.mapped_rmdup.pair.prim_rmdup.sort_rmdup.coord.bam \
  --output_vcf=/output/iceman.vcf \
  --output_gvcf=/output/iceman.gvcf \
  --num_shards=8 \
  --vcf_stats_report=true \
  --disable_small_model=false \
  --logging_dir=/output/logs \
  --haploid_contigs=chrX,chrY \
  --par_regions_bed=/input/GRCh38_PAR.bed
"""
print(DV_CMD)


## 6) Post-run Quick Stats

In [ ]:
run(f'bcftools stats {OUTPUT_DIR}/iceman.vcf > {OUTPUT_DIR}/iceman.vcf.stats')
run(f'tail -n 80 {OUTPUT_DIR}/iceman.vcf.stats')
